In [1]:
# 放在文件最顶部：在任何 sklearn / optuna 导入之前
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*sklearn\.utils\.parallel\.delayed.*Parallel.*",
    category=UserWarning
)

# -*- coding: utf-8 -*-
import re
import glob
import json
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

# 可选：保存最终模型
import joblib

# =========================
# 0) 配置 & 尽量屏蔽警告
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

FEATURE_FILE_GLOB = "./Malodors_StructKG_features.xlsx"

# 你 Optuna 搜索保存的结果
OPTUNA_RESULTS_CSV = "../optuna_singlemodel_RF_30trials_5fold_results.csv"

# 输出目录/文件
OUT_FOLDS_CSV = "./best_model_output/rf_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV = "./best_model_output/rf_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE = "./best_model_output/rf_bestparams_fullfit.joblib"

# RandomForest 固定参数
BASE_RF_PARAMS = dict(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

# =========================
# 1) 读特征文件：固定分配 X/y
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def load_Xy_from_transformed_morgan(xlsx_path: str):
    """
    新气味数据集列分配方式：
    - 第 1 列：SMILES / StdSMILES
    - 第 2–25 列：24 个气味描述符，作为 y
    - 第 26 列到最后：全部分子特征，作为 X

    注意：
    这里不再自动识别 0/1 标签列，避免把 FG、Morgan bit、Rule 等二值特征误判为标签。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件只有 {df.shape[1]} 列，但至少需要 26 列："
            f"第1列SMILES，第2–25列为24个气味描述符，第26列起为特征。"
        )

    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别到 {len(label_cols)} 个：{label_cols}"
        )

    if len(feature_cols) == 0:
        raise ValueError("未识别到特征列。请确认第 26 列之后为分子特征。")

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] SMILES column:", smiles_col)
    print("[INFO] Label columns:", label_cols)
    print("[INFO] First 5 feature columns:", feature_cols[:5])

    return X, y, feature_cols, label_cols, df

# =========================
# 2) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int, classes_list=None):
    # multi-output RF: list length=L, each (n, n_classes_k)
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
        for k in range(n_labels):
            pk = p[k]
            if pk.shape[1] == 2:
                if classes_list is not None and len(classes_list) == n_labels:
                    cls = list(classes_list[k])
                    if 1 in cls:
                        out[:, k] = pk[:, cls.index(1)].astype(np.float32)
                    else:
                        out[:, k] = 0.0
                else:
                    out[:, k] = pk[:, 1].astype(np.float32)
            elif pk.shape[1] == 1:
                if classes_list is not None and len(classes_list) == n_labels:
                    only_cls = int(list(classes_list[k])[0])
                    out[:, k] = 1.0 if only_cls == 1 else 0.0
                else:
                    out[:, k] = 0.0
            else:
                raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
        return out
    raise ValueError("RF multi-output predict_proba 预期返回 list，但未得到 list。")

# =========================
# 3) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 4) 从 Optuna 结果 CSV 读取最优超参
# =========================
def _maybe_to_number(x):
    # 兼容 "None" / "nan" / 数字字符串
    if pd.isna(x):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s.lower() in ["none", "null"]:
            return None
        if s.lower() in ["true", "false"]:
            return s.lower() == "true"
        try:
            if re.fullmatch(r"-?\d+", s):
                return int(s)
            if re.fullmatch(r"-?\d+(\.\d+)?([eE]-?\d+)?", s):
                return float(s)
        except Exception:
            return x
    return x

def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = set([
        "trial",
        "AUPRC_macro",
        "AUROC_macro",
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
    ])

    params = {}
    for k, v in best_row.items():
        if k in drop_cols:
            continue
        params[k] = _maybe_to_number(v)

    # 清理 max_depth=0 => None
    if "max_depth" in params and params["max_depth"] == 0:
        params["max_depth"] = None

    final_params = dict(BASE_RF_PARAMS)
    final_params.update(params)

    # 一些 max_features 可能是字符串形式的数字
    if "max_features" in final_params and isinstance(final_params["max_features"], str):
        mf = final_params["max_features"]
        if mf not in ["sqrt", "log2"]:
            try:
                final_params["max_features"] = float(mf)
            except Exception:
                pass

    return final_params

# =========================
# 5) 五折CV评估（用固定最优超参）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = RandomForestClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(
            p,
            n_labels=n_labels,
            classes_list=getattr(clf, "classes_", None)
        )

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    folds_df = pd.DataFrame(fold_rows).sort_values("fold")
    return folds_df

# =========================
# 6) 统计 mean ± CI95
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n - 1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_ci95(mean, lo, hi):
    """
    最终输出保留三位小数。
    """
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.3f} ± {half:.3f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean ± CI95": format_mean_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 7) 主流程
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(
        f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | "
        f"y shape={y.shape} (labels={len(label_cols)})"
    )

    # 读取最优超参
    best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)
    print("\n[INFO] Loaded best params from Optuna CSV:")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    # 五折
    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    # CV评估
    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)

    # 保存每折指标
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    # mean ± CI95 汇总
    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN ± CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean ± CI95']}")

    # 可选：用全量数据训练最终模型并保存
    clf_full = RandomForestClassifier(**best_params)
    clf_full.fit(X, y)
    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS


[INFO] Using feature file: ./Malodors_StructKG_features.xlsx
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: ['alcoholic', 'aldehydic', 'almond', 'aromatic', 'burnt', 'cabbage', 'cheesy', 'cherry', 'chocolate', 'ethereal', 'fishy', 'fruity', 'garlic', 'gassy', 'green', 'ketonic', 'musty', 'pungent', 'sharp', 'solvent', 'sour', 'sulfurous', 'sweaty', 'sweet']
[INFO] First 5 feature columns: ['KG_Element__Br', 'KG_Element__C', 'KG_Element__Cl', 'KG_Element__N', 'KG_Element__O']
[INFO] X shape=(3756, 168) (features=168) | y shape=(3756, 24) (labels=24)

[INFO] Loaded best params from Optuna CSV:
{
  "n_jobs": -1,
  "random_state": 42,
  "max_depth": 8,
  "n_estimators": 1003,
  "max_features": 1.0,
  "min_samples_split": 9,
  "min_samples_leaf": 4,
  "bootstrap": true,
  "criterion": "entropy"
}

[INFO] Prepared fixed 5-fold splits.
[FOLD 1] AUPRC=0.328481 | AUROC=0.842909 | Acc=0.938719 | P=0.695931 | R=0.165173 | Spec=0.975177
[FOLD 2] AUPRC=0.302162 | AUROC=0.836501 | Acc=

In [2]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier

# 可选：保存最终模型
import joblib

# =========================
# 0) 全局：尽量屏蔽警告 + LightGBM 日志
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError(
        "未检测到 lightgbm。请先安装：pip install lightgbm\n"
        f"原始错误：{repr(e)}"
    )

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

FEATURE_FILE_GLOB = "./Malodors_StructKG_features.xlsx"
OPTUNA_RESULTS_CSV = "../optuna_singlemodel_LIGHTGBM_30trials_5fold_results.csv"

OUT_FOLDS_CSV = "./best_model_output/lgb_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV = "./best_model_output/lgb_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE = "./best_model_output/lgb_bestparams_fullfit.joblib"

# 固定参数（会与 best_params 合并）
BASE_LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,  # 关闭 LightGBM 日志
)

# =========================
# 2) 读取 transformed 特征文件 & 固定分配 X/y
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def load_Xy_from_transformed_morgan(xlsx_path: str):
    """
    新气味数据集列分配方式：
    - 第 1 列：SMILES / StdSMILES
    - 第 2–25 列：24 个气味描述符，作为 y
    - 第 26 列到最后：全部分子特征，作为 X

    注意：
    这里不再自动识别 0/1 标签列，避免把 FG、Morgan bit、Rule 等二值特征误判为标签。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件只有 {df.shape[1]} 列，但至少需要 26 列："
            f"第1列SMILES，第2–25列为24个气味描述符，第26列起为特征。"
        )

    # 第 1 列：SMILES / StdSMILES
    smiles_col = df.columns[0]

    # 第 2–25 列：24 个气味描述符，Python 对应 1:25
    label_cols = list(df.columns[1:25])

    # 第 26 列到最后：特征，Python 对应 25:
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别到 {len(label_cols)} 个：{label_cols}"
        )

    if len(feature_cols) == 0:
        raise ValueError("未识别到特征列。请确认第 26 列之后为分子特征。")

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] SMILES column:", smiles_col)
    print("[INFO] Label columns:", label_cols)
    print("[INFO] First 5 feature columns:", feature_cols[:5])

    return X, y, feature_cols, label_cols, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int, classes_list=None):
    """
    MultiOutputClassifier.predict_proba -> list length=L
    兼容：某些标签在某 fold 训练集只有单类 => (n,1)
    """
    if not isinstance(p, list):
        raise ValueError("MultiOutputClassifier.predict_proba 预期返回 list，但未得到 list。")

    out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
    for k in range(n_labels):
        pk = p[k]
        if pk.ndim != 2:
            raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

        if pk.shape[1] == 2:
            if classes_list is not None and len(classes_list) == n_labels:
                cls = list(classes_list[k])
                if 1 in cls:
                    out[:, k] = pk[:, cls.index(1)].astype(np.float32)
                else:
                    out[:, k] = 0.0
            else:
                out[:, k] = pk[:, 1].astype(np.float32)

        elif pk.shape[1] == 1:
            if classes_list is not None and len(classes_list) == n_labels:
                only_cls = int(list(classes_list[k])[0])
                out[:, k] = 1.0 if only_cls == 1 else 0.0
            else:
                out[:, k] = 0.0
        else:
            raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
    return out

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 从 Optuna 结果 CSV 读取最优超参 + 类型修复
# =========================
def sanitize_lgb_params(params: dict) -> dict:
    """
    修复从 CSV 读出来的 best_params 类型问题：
    - n_estimators / num_leaves / max_depth / min_child_samples / n_jobs / random_state 必须 int
    - 其它本应 float 的保持 float
    """
    p = dict(params)

    int_keys = [
        "n_estimators",
        "num_leaves",
        "max_depth",
        "min_child_samples",
        "n_jobs",
        "random_state",
        "verbosity",
    ]
    for k in int_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = int(float(p[k]))
            except Exception:
                pass

    float_keys = [
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
        "reg_alpha",
        "min_split_gain",
    ]
    for k in float_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = float(p[k])
            except Exception:
                pass

    return p

def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = set([
        "trial",
        "AUPRC_macro",
        "AUROC_macro",
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
    ])

    params = {}
    for k, v in best_row.items():
        if k in drop_cols:
            continue
        params[k] = v

    final_params = dict(BASE_LGB_PARAMS)
    final_params.update(params)

    final_params = sanitize_lgb_params(final_params)

    return final_params

# =========================
# 6) 五折CV评估（用固定最优超参）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = lgb.LGBMClassifier(**params)
        clf = MultiOutputClassifier(base_est, n_jobs=1)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(
            p,
            n_labels=n_labels,
            classes_list=getattr(clf, "classes_", None)
        )

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    return pd.DataFrame(fold_rows).sort_values("fold")

# =========================
# 7) mean ± CI95（小样本用t；无scipy则退回1.96）
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n - 1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_pm_ci95(mean, lo, hi):
    """
    最终输出保留三位小数。
    """
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.3f} ± {half:.3f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean ± CI95": format_mean_pm_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 8) 主流程：读取 -> 读最优参数 -> 5折CV -> 输出&保存 -> 全量fit保存
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(
        f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | "
        f"y shape={y.shape} (labels={len(label_cols)})"
    )

    best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)
    print("\n[INFO] Loaded best params from Optuna CSV (sanitized):")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN ± CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean ± CI95']}")

    # 可选：用全量数据训练最终模型并保存
    base_est = lgb.LGBMClassifier(**best_params)
    clf_full = MultiOutputClassifier(base_est, n_jobs=1)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS


[INFO] Using transformed feature file: ./Malodors_StructKG_features.xlsx
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: ['alcoholic', 'aldehydic', 'almond', 'aromatic', 'burnt', 'cabbage', 'cheesy', 'cherry', 'chocolate', 'ethereal', 'fishy', 'fruity', 'garlic', 'gassy', 'green', 'ketonic', 'musty', 'pungent', 'sharp', 'solvent', 'sour', 'sulfurous', 'sweaty', 'sweet']
[INFO] First 5 feature columns: ['KG_Element__Br', 'KG_Element__C', 'KG_Element__Cl', 'KG_Element__N', 'KG_Element__O']
[INFO] X shape=(3756, 168) (features=168) | y shape=(3756, 24) (labels=24)

[INFO] Loaded best params from Optuna CSV (sanitized):
{
  "objective": "binary",
  "boosting_type": "gbdt",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": -1,
  "n_estimators": 1534,
  "learning_rate": 0.1258550866043942,
  "num_leaves": 175,
  "max_depth": 13,
  "min_child_samples": 42,
  "subsample": 0.7730982444721965,
  "colsample_bytree": 0.6072358215492547,
  "reg_lambda": 0.0055234244890457,
  "reg_al

In [3]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# 可选：保存最终模型
import joblib

# =========================
# 0) 全局：尽量屏蔽警告
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_StructKG_features.xlsx"

# 你 Optuna 搜索输出的结果文件（用于读取 best params）
OPTUNA_RESULTS_CSV = "../optuna_singlemodel_XGB_30trials_5fold_results.csv"

# 输出文件
OUT_FOLDS_CSV = "./best_model_output/xgb_bestparams_5fold_metrics.csv"
OUT_MEANCI_CSV = "./best_model_output/xgb_bestparams_5fold_mean_ci95.csv"
OUT_MODEL_FILE = "./best_model_output/xgb_bestparams_fullfit.joblib"

# 固定参数（会与 best_params 合并）
BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",  # 需要 xgboost>=2.0
)

# =========================
# 2) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")
    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree（需要>=2.0）。"
            f"升级或删掉 multi_strategy。"
        )

# =========================
# 3) 读取 transformed 特征文件 & 固定分配 X/y
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def load_Xy_from_transformed_morgan(xlsx_path: str):
    """
    新气味数据集列分配方式：
    - 第 1 列：SMILES / StdSMILES
    - 第 2–25 列：24 个气味描述符，作为 y
    - 第 26 列到最后：全部分子特征，作为 X

    注意：
    这里不再自动识别 0/1 标签列，避免把 Morgan bit、FG、Rule 等二值特征误判为标签。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件只有 {df.shape[1]} 列，但至少需要 26 列："
            f"第1列SMILES，第2–25列为24个气味描述符，第26列起为特征。"
        )

    # 第 1 列：SMILES / StdSMILES
    smiles_col = df.columns[0]

    # 第 2–25 列：24 个气味描述符，Python 对应 1:25
    label_cols = list(df.columns[1:25])

    # 第 26 列到最后：特征，Python 对应 25:
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别到 {len(label_cols)} 个：{label_cols}"
        )

    if len(feature_cols) == 0:
        raise ValueError("未识别到特征列。请确认第 26 列之后为分子特征。")

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] SMILES column:", smiles_col)
    print("[INFO] Label columns:", label_cols)
    print("[INFO] First 5 feature columns:", feature_cols[:5])

    return X, y, feature_cols, label_cols, df

# =========================
# 4) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    """
    multi-output XGBClassifier predict_proba 可能是：
    - ndarray (n, L)         : 已是正类概率
    - ndarray (n, L, 2)      : 二分类概率
    - list length=L, (n,2)   : 每标签一组概率
    """
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 5) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 6) 从 Optuna 结果 CSV 读取最优超参 + 类型修复
# =========================
def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = set([
        "trial", "AUPRC_macro",
        "AUROC_macro", "Accuracy_macro", "Precision_macro", "Recall_macro", "Specificity_macro",
    ])

    params = {}
    for k, v in best_row.items():
        if k in drop_cols:
            continue
        params[k] = v

    final_params = dict(BASE_XGB_PARAMS)
    final_params.update(params)
    final_params = sanitize_xgb_params(final_params)

    return final_params

def sanitize_xgb_params(params: dict) -> dict:
    """
    修复从 CSV 读出来的 best_params 类型问题。
    """
    p = dict(params)

    int_keys = ["n_estimators", "max_depth", "n_jobs", "random_state"]
    for k in int_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = int(float(p[k]))
            except Exception:
                pass

    float_keys = [
        "learning_rate", "subsample", "colsample_bytree",
        "min_child_weight", "reg_lambda", "reg_alpha", "gamma"
    ]
    for k in float_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = float(p[k])
            except Exception:
                pass

    return p

# =========================
# 7) 五折CV评估（用固定最优超参）
# =========================
def cv_eval_bestparams(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        m["fold"] = fold_id
        fold_rows.append(m)

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | Spec={m['Specificity_macro']:.6f}"
        )

    return pd.DataFrame(fold_rows).sort_values("fold")

# =========================
# 8) mean ± CI95
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n - 1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_pm_ci95(mean, lo, hi):
    """
    最终输出保留三位小数。
    """
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.3f} ± {half:.3f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]
    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean ± CI95": format_mean_pm_ci95(mean, lo, hi),
        })
    return pd.DataFrame(rows)

# =========================
# 9) 主流程：读特征 -> 读最优参数 -> 5折CV -> 输出&保存 -> 全量fit保存
# =========================
def main():
    check_xgb_version()

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed Morgan feature file:", feature_file)

    X, y, feat_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)
    print("\n[INFO] Loaded best params from Optuna CSV (sanitized):")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    # 5-fold CV
    folds_df = cv_eval_bestparams(X, y, folds, best_params, thresh=THRESH)
    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] {OUT_FOLDS_CSV}")

    # mean ± CI95 汇总
    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN ± CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean ± CI95']}")

    # 全量训练并保存
    clf_full = xgb.XGBClassifier(**best_params)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feat_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
        },
        OUT_MODEL_FILE
    )
    print(f"\n[SAVED] full-fit model -> {OUT_MODEL_FILE}")

if __name__ == "__main__":
    main()

[INFO] Using transformed Morgan feature file: ./Malodors_StructKG_features.xlsx
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: ['alcoholic', 'aldehydic', 'almond', 'aromatic', 'burnt', 'cabbage', 'cheesy', 'cherry', 'chocolate', 'ethereal', 'fishy', 'fruity', 'garlic', 'gassy', 'green', 'ketonic', 'musty', 'pungent', 'sharp', 'solvent', 'sour', 'sulfurous', 'sweaty', 'sweet']
[INFO] First 5 feature columns: ['KG_Element__Br', 'KG_Element__C', 'KG_Element__Cl', 'KG_Element__N', 'KG_Element__O']
[INFO] X shape=(3756, 168) (features=168) | y shape=(3756, 24) (labels=24)

[INFO] Loaded best params from Optuna CSV (sanitized):
{
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "tree_method": "hist",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": 0,
  "multi_strategy": "multi_output_tree",
  "n_estimators": 1171,
  "max_depth": 7,
  "learning_rate": 0.0230049001803117,
  "subsample": 0.9995045229260104,
  "colsample_bytree": 0.8382475684166885,
  "min_child